# SatQKD-Sentinel 2.0.2 — Google Colab

**Завантажте лише цей `.ipynb` у Google Colab і виберіть Runtime → Run all (Середовище виконання → Виконати всі).** CPU достатньо; GPU не потрібний. Увесь код і тести вже вбудовані. Немає `../`, `git clone`, завантаження додаткового ZIP або підключення Drive.

Виконуйте комірки зверху вниз. Після скидання сеансу знову запустіть **Run all**. Наприкінці виконуються всі автоматичні тести, створюється ZIP результатів і з’являється кнопка завантаження. Перша комірка встановлює лише відсутні залежності; потрібен доступ до PyPI, якщо їх немає.

Це дослідницький симулятор на синтетичних даних. Finite-size оцінка не є доказом криптографічної безпеки.


In [ ]:
# 1. Самодостатнє налаштування Google Colab / Jupyter. CPU достатньо.
import base64, hashlib, importlib, importlib.metadata, io, os, subprocess, sys, tempfile, zipfile
from pathlib import Path
if sys.version_info < (3, 10):
    raise RuntimeError("Потрібен Python 3.10 або новіший.")
# Do not upgrade Colab's preinstalled scientific stack unnecessarily.
requirements = {
    "numpy": "numpy>=1.26", "scipy": "scipy>=1.12", "pandas": "pandas>=2.1",
    "matplotlib": "matplotlib>=3.8", "sklearn": "scikit-learn>=1.4",
    "joblib": "joblib>=1.3", "pytest": "pytest>=8", "packaging": "packaging>=23",
    "ipywidgets": "ipywidgets>=7.7",
}
missing = [spec for module, spec in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
from packaging.requirements import Requirement
outdated = [spec for spec in requirements.values()
            if importlib.metadata.version(Requirement(spec).name) not in Requirement(spec).specifier]
if outdated:
    raise RuntimeError("Застарілі залежності: " + ", ".join(outdated) +
                       ". Оновіть їх через %pip install, перезапустіть сеанс і виконайте всі комірки.")
_SOURCE_ZIP = "..."
_SOURCE_SHA256 = "d3c10999738fd442d8bf75dd498e63f93b0d37d6e4930aa5810616281e1d8994"
raw = base64.b64decode(_SOURCE_ZIP, validate=True)
assert hashlib.sha256(raw).hexdigest() == _SOURCE_SHA256, "Embedded source checksum mismatch"
work_parent = Path("/content") if Path("/content").is_dir() else Path.cwd()
PROJECT_ROOT = Path(tempfile.mkdtemp(prefix="satqkd_2_0_2_", dir=work_parent)).resolve()
with zipfile.ZipFile(io.BytesIO(raw)) as source_zip:
    for member in source_zip.namelist():
        if Path(member).is_absolute() or ".." in Path(member).parts:
            raise ValueError("Unsafe embedded archive path")
    source_zip.extractall(PROJECT_ROOT)
# Rerunning this cell must never reuse another installed version of the project.
for name in list(sys.modules):
    if name == "satqkd_sentinel" or name.startswith("satqkd_sentinel."):
        del sys.modules[name]
sys.path.insert(0, str(PROJECT_ROOT))
importlib.invalidate_caches()
import satqkd_sentinel
assert Path(satqkd_sentinel.__file__).resolve().is_relative_to(PROJECT_ROOT)
assert satqkd_sentinel.__version__ == "2.0.2"
OUTPUT_DIR = PROJECT_ROOT / "results"
OUTPUT_DIR.mkdir()
print("Готово: SatQKD-Sentinel", satqkd_sentinel.__version__)
print("Код відновлено з ноутбука:", PROJECT_ROOT)
print("GPU, Google Drive та додаткові файли не потрібні.")


In [ ]:
%matplotlib inline
import sys, os, warnings
assert PROJECT_ROOT.is_dir(), "Спочатку виконайте комірку налаштування."


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import satqkd_sentinel
from satqkd_sentinel.linkmodel import Link, slant_range, pass_profile, h2, R_E, H_GEO
from satqkd_sentinel.finitekey import finite_key_pass, finite_key_length, _pass_counts
from satqkd_sentinel.opa import (array_factor_1d, beam_pattern, steering_range,
                                  beam_width, phase_error_loss_db, sidelobe_level_db,
                                  far_field_divergence)
from satqkd_sentinel.geo import (coverage_angle, coverage_stats, isl_chord,
                                  contact_plan, max_gap_s)
from satqkd_sentinel.network import (key_per_pass, key_per_day, daily_key_budget,
                                      key_store_model, network_availability,
                                      relay_key_consumption, cost_model)
from satqkd_sentinel.telemetry import generate_pass, generate_dataset, STATES, GROUPS
from satqkd_sentinel.features import extract_features, FEATURE_NAMES, feature_dictionary
from satqkd_sentinel.models import (train_classifier, evaluate_classifier, predict,
                                     train_novelty_detector, novelty_predict,
                                     feature_importance, block_policy, window_alarm)
from satqkd_sentinel.evaluation import (full_evaluation, summary_table,
                                         static_threshold_baseline,
                                         leave_one_attack_out)

print(f"satqkd_sentinel v{satqkd_sentinel.__version__}")
print(f"NumPy {np.__version__}, pandas {pd.__version__}")
# Capture every displayed figure as a PNG, while preserving inline display.
_original_show = getattr(plt.show, "_satqkd_original_show", plt.show)
_plot_counter = 0

def _show_and_save(*args, **kwargs):
    global _plot_counter
    for number in plt.get_fignums():
        _plot_counter += 1
        plt.figure(number).savefig(OUTPUT_DIR / f"figure_{_plot_counter:02d}.png", dpi=150, bbox_inches="tight")
    return _original_show(*args, **kwargs)

_show_and_save._satqkd_original_show = _original_show
plt.show = _show_and_save


# SatQKD-Sentinel v2.0.2 — Comprehensive Demonstration

This notebook provides a self-contained demonstration of all modules in the **satqkd_sentinel** package — a Python toolkit for modelling and simulating satellite quantum key distribution (QKD) links over geostationary backbone networks.

### Modules covered

| Module | Purpose |
|---|---|
| `linkmodel` | Free-space optical channel model, BB84 key rate (Eqs. 1–9) |
| `finitekey` | Illustrative finite-size estimate; not a composable security proof |
| `opa` | Optical Phased Array beam steering and phase-error loss (Eqs. 12–16) |
| `geo` | Geostationary constellation coverage and contact planning (Eqs. 28–33) |
| `network` | Trusted-relay key budget, key-store dynamics, availability |
| `telemetry` | Synthetic telemetry for 10 link states (1 nominal + 5 faults + 4 attacks) |
| `features` | 24 physics-informed feature extraction |
| `models` | ML classifiers (GB, RF, LR) and novelty detectors (IF, Mahalanobis) |
| `evaluation` | End-to-end evaluation framework with summary metrics |

---
## 1. Link Model — Free-Space BB84 Channel

The `linkmodel` module implements the end-to-end channel transmittance $\eta$ for a satellite-to-ground QKD link operating at 1550 nm. The total transmittance combines geometric divergence (Eq. 3), pointing error (Eq. 4), atmospheric extinction (Eq. 6), main-lobe efficiency, and receiver optics:

$$\eta = \eta_{\mathrm{geo}} \cdot \eta_{\mathrm{pt}} \cdot \eta_{\mathrm{atm}} \cdot \eta_{\mathrm{ml}} \cdot \eta_{\mathrm{rx}}$$

The asymptotic BB84 key rate (Eq. 9) then depends on the signal gain $Q_\mu$, quantum bit error rate $E_\mu$, and the single-photon yield and error rate estimated via the decoy-state method.

In [ ]:
# Default link parameters and link budget at selected elevations
link = Link()
print("=== Baseline Link Parameters ===")
params = {
    'Wavelength': f'{link.wavelength*1e9:.0f} nm',
    'TX aperture': f'{link.d_tx*100:.0f} cm',
    'RX aperture': f'{link.d_rx*100:.0f} cm',
    'Pointing error': f'{link.sigma_j*1e6:.1f} µrad',
    'Pulse rate': f'{link.f_rep/1e6:.0f} MHz',
    'Signal intensity µ': f'{link.mu}',
    'X-basis probability': f'{link.p_x}',
    'Optical error e_d': f'{link.e_d}',
    'Zenith transmittance': f'{link.t_zen}',
}
for k, v in params.items():
    print(f"  {k:25s} {v}")

print("\n=== Link Budget at Selected Elevations ===")
rows = []
for eps_deg in [20, 30, 45, 60, 90]:
    eps_rad = np.radians(eps_deg)
    L = slant_range(eps_rad, 500e3)
    comp = link.components_db(L, eps_rad)
    total = link.loss_db(L, eps_rad)
    rows.append(dict(elevation_deg=eps_deg, slant_range_km=L/1e3,
                     total_loss_dB=total, **comp))
pd.DataFrame(rows).round(1)

In [ ]:
# Total loss and component breakdown vs elevation
elevs = np.linspace(20, 90, 200)
elevs_rad = np.radians(elevs)
Ls = slant_range(elevs_rad, 500e3)

total_loss = link.loss_db(Ls, elevs_rad)
comp_names = ['diffraction', 'pointing', 'atmosphere', 'lobes', 'receiver']
comp_data = {c: [] for c in comp_names}
for e, l in zip(elevs_rad, Ls):
    cd = link.components_db(l, e)
    for c in comp_names:
        comp_data[c].append(cd[c])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(elevs, total_loss, 'k-', lw=2.5, label='Total loss')
colors = ['#2196F3', '#FF9800', '#9C27B0', '#FFC107', '#4CAF50']
for c, col in zip(comp_names, colors):
    ax.plot(elevs, comp_data[c], '--', color=col, lw=1.5, label=c.capitalize())
ax.set_xlabel('Elevation (deg)')
ax.set_ylabel('Loss (dB)')
ax.set_title('Channel Loss Budget vs Elevation (h = 500 km)')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(20, 90)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# Asymptotic key rate and QBER vs elevation
etas = link.eta(Ls, elevs_rad)
rates, qbers = link.rate(etas)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.semilogy(elevs, rates, color='#2196F3', lw=2, label='Key rate (bit/s)')
ax1.set_xlabel('Elevation (deg)')
ax1.set_ylabel('Key rate (bit/s)', color='#2196F3')
ax1.tick_params(axis='y', labelcolor='#2196F3')
ax1.set_xlim(20, 90)
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(elevs, qbers * 100, color='#F44336', lw=2, ls='--', label='QBER (%)')
ax2.set_ylabel('QBER (%)', color='#F44336')
ax2.tick_params(axis='y', labelcolor='#F44336')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
ax1.set_title('Asymptotic BB84 Key Rate and QBER vs Elevation')
plt.tight_layout()
plt.show()
plt.close()

---
## 2. Satellite Pass Profile & Key Generation

A LEO satellite at 500 km altitude traces an arc across the sky. The `pass_profile()` function generates the time-stamped elevation, slant range, and azimuth for a pass with a given maximum elevation $\varepsilon_{\max}$. The key generated per pass is the integral of the instantaneous rate over the visible arc above 20° minimum elevation.

In [ ]:
# Pass profile for eps_max = 40°
t, el, L, az = pass_profile(np.radians(40))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(t, np.degrees(el), color='#2196F3', lw=2)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Elevation (deg)')
ax1.set_title(f'Pass Profile (ε_max = 40°, duration = {t[-1]-t[0]:.0f} s)')
ax1.grid(True, alpha=0.3)
ax1.axhline(20, ls=':', color='gray', label='Min elevation 20°')
ax1.legend()

ax2.plot(t, L / 1e3, color='#FF9800', lw=2)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Slant range (km)')
ax2.set_title('Slant Range During Pass')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

print(f"Pass duration: {t[-1]-t[0]:.0f} s")
print(f"Min slant range: {L.min()/1e3:.0f} km (at zenith approach)")
print(f"Max slant range: {L.max()/1e3:.0f} km (at horizon)")

In [ ]:
# Asymptotic key per pass for all elevation maxima
eps_maxes = [25, 30, 40, 60, 90]
rows = []
for ed in eps_maxes:
    er = np.radians(ed)
    tp, elp, Lp, _ = pass_profile(er)
    k = key_per_pass(link, er)
    dur = tp[-1] - tp[0]
    rows.append(dict(eps_max_deg=ed, duration_s=dur, key_Mbit=k/1e6,
                     avg_rate_kbps=k/dur/1e3))
df_key = pd.DataFrame(rows)
print("=== Asymptotic Key per Pass ===")
df_key.round(3)

---
## 3. Illustrative Finite-Size Estimate

The `finitekey` module approximates a three-intensity scenario (signal 0.5, weak decoy 0.1, near-vacuum 0.001). It includes simplified gain fluctuations, observed signal errors, error correction and finite-size penalties. `block_size_s` partitions the pass and splits the epsilon budget between blocks.

**This is not a composable security proof or a deployable key generator.** The finite-size routine uses 50/25/25% pulse allocation, while the asymptotic reference uses all pulses as signals. The ratio therefore combines allocation and finite-size effects. Read the calculated table below rather than assuming a fixed penalty.

In [ ]:
# Finite vs asymptotic key comparison
rows = []
for ed in eps_maxes:
    er = np.radians(ed)
    k_asym = key_per_pass(link, er)
    k_fin = finite_key_pass(link, er)
    ratio = k_fin / k_asym if k_asym > 0 else 0
    rows.append(dict(eps_max_deg=ed, asymptotic_Mbit=k_asym/1e6,
                     finite_Mbit=k_fin/1e6, ratio=ratio))

df_fk = pd.DataFrame(rows)
print("=== Asymptotic vs Finite Key per Pass ===")
df_fk.round(4)

In [ ]:
# Bar chart: asymptotic vs finite key
x = np.arange(len(eps_maxes))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, df_fk['asymptotic_Mbit'], width,
               label='Asymptotic', color='#2196F3', alpha=0.8)
bars2 = ax.bar(x + width/2, df_fk['finite_Mbit'], width,
               label='Finite-key', color='#FF9800', alpha=0.8)

ax.set_xlabel('Maximum Elevation (deg)')
ax.set_ylabel('Key per Pass (Mbit)')
ax.set_title('Asymptotic vs Finite-Key Secret Key Length')
ax.set_xticks(x)
ax.set_xticklabels([f'{e}°' for e in eps_maxes])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for b, r in zip(bars2, df_fk['ratio']):
    ax.annotate(f'{r:.1%}', xy=(b.get_x() + b.get_width()/2, b.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# Detection statistics underlying the finite-key bound (90° pass)
c = _pass_counts(link, np.radians(90))
print("=== Detection Statistics (ε_max = 90°) ===")
for name in ('mu', 'nu1', 'nu2'):
    d = c[name]
    Q = d['n_det'] / d['n_sent']
    print(f"  {name:4s}: N_sent = {d['n_sent']:.2e}, "
          f"N_det = {d['n_det']:.2e}, Q = {Q:.2e}, E = {d['e_avg']:.4f}")

---
## 4. Optical Phased Array (OPA) Beam Steering

The model uses 4096 elements with a 2 µm pitch at 1550 nm. This is larger than one wavelength (half-wavelength would be 0.775 µm), so visible grating lobes exist. The approximately ±22.8° quantity is the principal spatial-frequency interval, not a grating-lobe-free guarantee. `beam_width` returns the peak-to-first-null angle; `far_field_divergence` returns the approximate full FWHM. Gaussian phase-error loss is modeled separately.

In [ ]:
# OPA key metrics
sr = steering_range()
bw = beam_width()
sl = sidelobe_level_db()
ffd = far_field_divergence()
pe04 = phase_error_loss_db(0.4)
pe10 = phase_error_loss_db(1.0)

opa_data = pd.DataFrame([
    dict(metric='Steering half-angle', value=f'±{np.degrees(sr):.1f}°'),
    dict(metric='Peak to first null', value=f'{bw*1e6:.0f} µrad'),
    dict(metric='Full FWHM', value=f'{ffd*1e6:.0f} µrad'),
    dict(metric='First sidelobe level', value=f'{sl:.1f} dB'),
    dict(metric='Phase-error loss (σ=0.4 rad)', value=f'{pe04:.2f} dB'),
    dict(metric='Phase-error loss (σ=1.0 rad)', value=f'{pe10:.1f} dB'),
])
print("=== OPA Performance Metrics ===")
opa_data

In [ ]:
# Beam pattern: broadside vs 10° steering
theta = np.linspace(np.radians(-30), np.radians(30), 5000)

af_broad = array_factor_1d(theta, 0.0)
af_steer = array_factor_1d(theta, np.radians(10))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.degrees(theta), 10*np.log10(np.clip(af_broad, 1e-10, None)),
        color='#2196F3', lw=1.5, label='Broadside (θ_s = 0°)')
ax.plot(np.degrees(theta), 10*np.log10(np.clip(af_steer, 1e-10, None)),
        color='#F44336', lw=1.5, label='Steered (θ_s = 10°)')
ax.set_xlabel('Observation angle θ (deg)')
ax.set_ylabel('Normalised pattern (dB)')
ax.set_title('OPA 1-D Array Factor (N = 4096, d = 2 µm)')
ax.set_ylim(-40, 1)
ax.set_xlim(-30, 30)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# Phase-error loss vs standard deviation
sigmas = np.linspace(0.01, 1.5, 100)
losses = [phase_error_loss_db(s) for s in sigmas]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sigmas, losses, color='#9C27B0', lw=2)
ax.axvline(0.4, ls='--', color='#FF9800', label=f'σ = 0.4 rad → {pe04:.2f} dB')
ax.axvline(1.0, ls='--', color='#F44336', label=f'σ = 1.0 rad → {pe10:.1f} dB')
ax.set_xlabel('Phase error σ (rad)')
ax.set_ylabel('On-axis power loss (dB)')
ax.set_title('OPA Phase-Error Strehl Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

---
## 5. Geostationary Backbone Coverage

The `geo` module analyses equatorial GEO constellations as the backbone for a global QKD network. It computes the coverage footprint (the zone of the Earth's surface that sees at least one node above 20° elevation), inter-satellite link (ISL) distances, and the contact plan between a LEO relay satellite and the GEO ring plus ground stations.

In [ ]:
# GEO constellation coverage for n = 3, 4, 6 nodes
rows = []
for n in [3, 4, 6]:
    cs = coverage_stats(n, eps_min_deg=20.0, nlat=181, nlon=361)
    rows.append(dict(n_nodes=n,
                     single_coverage_pct=cs['share1']*100,
                     dual_coverage_pct=cs['share2']*100,
                     lat_continuous_deg=cs['lat_cont']))
df_cov = pd.DataFrame(rows)
print("=== GEO Constellation Coverage (ε_min = 20°) ===")
df_cov.round(1)

In [ ]:
# Inter-satellite link (ISL) chord distances
rows = []
for n in [3, 4, 6, 8]:
    d = isl_chord(n)
    rows.append(dict(n_nodes=n, isl_distance_km=d/1e3))
df_isl = pd.DataFrame(rows)
print("=== ISL Chord Distance Between Adjacent GEO Nodes ===")
df_isl.round(0)

In [ ]:
# Contact plan: LEO relay visibility from 4-node GEO ring and 2 ground stations (6 hours)
sites = [(48.8, 2.3), (35.7, 139.7)]  # Paris, Tokyo
cp = contact_plan(4, sites, hours=6.0, dt=30.0)

fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
t_hr = cp['t'] / 3600

for i in range(4):
    axes[0].fill_between(t_hr, i + 0.1, i + 0.9, where=cp['geo'][i],
                         alpha=0.6, label=f'GEO-{i+1}')
axes[0].set_ylabel('GEO node')
axes[0].set_yticks([0.5, 1.5, 2.5, 3.5])
axes[0].set_yticklabels(['GEO-1', 'GEO-2', 'GEO-3', 'GEO-4'])
axes[0].set_title('LEO Relay Visibility Contact Plan (6 hours)')

for i, name in enumerate(['Paris', 'Tokyo']):
    axes[1].fill_between(t_hr, i + 0.1, i + 0.9, where=cp['site'][i],
                         alpha=0.6, color=['#4CAF50', '#FF9800'][i], label=name)
axes[1].set_ylabel('Ground site')
axes[1].set_yticks([0.5, 1.5])
axes[1].set_yticklabels(['Paris', 'Tokyo'])

axes[2].plot(t_hr, cp['geo_range'][0] / 1e3, lw=1, label='Range to GEO-1')
axes[2].set_ylabel('Range (km)')
axes[2].set_xlabel('Time (hours)')
axes[2].legend()
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

---
## 6. Trusted-Relay Network & Key Budget

The `network` module computes the daily key budget for varying numbers of satellite passes, models key-store dynamics (generation vs consumption), availability of redundant independent links, and relay key consumption for multi-hop trusted-node forwarding.

In [ ]:
# Daily key budget for different numbers of passes
budget = daily_key_budget()
df_budget = pd.DataFrame([
    dict(passes_per_day=p, daily_key_Mbit=k/1e6)
    for p, k in sorted(budget.items())
])
print("=== Daily Key Budget (ε_max = 40°, asymptotic) ===")
df_budget.round(2)

In [ ]:
# Key-store dynamics simulation (30 days)
gen_rate = 5000  # 5 kbit/s generation during a pass
cons_rate = 100  # 100 bit/s continuous consumption
capacity = 50e6  # 50 Mbit store
ks = key_store_model(gen_rate, cons_rate, capacity, days=30)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ks['t_hours'] / 24, ks['store'] / 1e6, color='#2196F3', lw=1.5)
ax.axhline(capacity / 1e6, ls='--', color='#F44336', label='Capacity')
ax.fill_between(ks['t_hours'] / 24, 0, ks['store'] / 1e6, alpha=0.15, color='#2196F3')
ax.set_xlabel('Day')
ax.set_ylabel('Key store (Mbit)')
ax.set_title('Key-Store Dynamics Over 30 Days')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 30)
plt.tight_layout()
plt.show()
plt.close()

print(f"Overflow: {ks['overflow_bits']/1e6:.2f} Mbit, Underflow: {ks['underflow_bits']/1e6:.2f} Mbit")

In [ ]:
# Availability of independent links
rows = []
for n in [3, 4, 6, 8]:
    for p in [0.90, 0.95, 0.99]:
        a1 = network_availability(n, p, redundancy=1)
        a2 = network_availability(n, p, redundancy=2)
        rows.append(dict(n_nodes=n, link_avail=p, avail_1path=a1, avail_2path=a2))
df_avail = pd.DataFrame(rows)
print("=== Independent-Link Availability ===")
df_avail.round(4)

---
## 7. Telemetry Simulation

The `telemetry` module generates synthetic 1-second telemetry windows for **10 link states**: one nominal state, five hardware/environmental fault types, and four quantum-attack signatures. Each window includes quantum and classical observables (quantum statistics, beacon power, pointing, receiver monitors, etc.)

| Group | States |
|---|---|
| Nominal | `nominal` |
| Faults | `cloud_attenuation`, `pointing_degradation`, `opa_phase_drift`, `natural_background`, `polarization_drift` |
| Attacks | `optical_jamming`, `intercept_resend`, `yield_manipulation`, `detector_control` |

In [ ]:
# Generate and visualise a nominal pass
nom_pass = generate_pass(link, np.radians(40), state='nominal', seed=42)
print(f"Nominal pass: {len(nom_pass)} windows")
print(nom_pass[['t_s', 'elevation_deg', 'loss_db', 'qber', 'beacon_down', 'label']].head(10))

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
axes[0].plot(nom_pass['t_s'], nom_pass['loss_db'], color='#2196F3', lw=1)
axes[0].set_ylabel('Loss (dB)')
axes[0].set_title('Nominal Pass Telemetry (ε_max = 40°)')

axes[1].plot(nom_pass['t_s'], nom_pass['qber'], color='#4CAF50', lw=1)
axes[1].set_ylabel('QBER')

axes[2].plot(nom_pass['t_s'], nom_pass['beacon_down'], color='#FF9800', lw=1, label='Down')
axes[2].plot(nom_pass['t_s'], nom_pass['beacon_up'], color='#9C27B0', lw=1, label='Up')
axes[2].set_ylabel('Beacon power')
axes[2].set_xlabel('Time (s)')
axes[2].legend()

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# Anomalous pass: optical jamming with severity 0.7
jam_pass = generate_pass(link, np.radians(40), state='optical_jamming',
                         severity=0.7, onset_fraction=0.3, seed=123)

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
onset_t = jam_pass.loc[jam_pass['label'] == 'optical_jamming', 't_s'].iloc[0]

axes[0].plot(jam_pass['t_s'], jam_pass['loss_db'], color='#2196F3', lw=1)
axes[0].set_ylabel('Loss (dB)')
axes[0].set_title('Optical Jamming Attack (severity = 0.7, onset at 30%)')

axes[1].plot(jam_pass['t_s'], jam_pass['qber'], color='#4CAF50', lw=1)
axes[1].set_ylabel('QBER')

axes[2].plot(jam_pass['t_s'], jam_pass['vacuum_rate'], color='#F44336', lw=1, label='Vacuum rate')
axes[2].plot(jam_pass['t_s'], jam_pass['double_click_rate']*100, color='#9C27B0', lw=1, label='Double-click ×100')
axes[2].set_ylabel('Rate')
axes[2].set_xlabel('Time (s)')
axes[2].legend()

for ax in axes:
    ax.axvline(onset_t, ls='--', color='red', alpha=0.5, lw=1)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# Generate multi-class dataset (30 passes)
dataset = generate_dataset(link, 30, seed=42)
print(f"Dataset shape: {dataset.shape}")
print(f"\n=== Class Distribution ===")
class_counts = dataset['label'].value_counts()
print(class_counts)

fig, ax = plt.subplots(figsize=(10, 4))
colors_map = {'nominal': '#4CAF50',
              'cloud_attenuation': '#2196F3', 'pointing_degradation': '#03A9F4',
              'opa_phase_drift': '#00BCD4', 'natural_background': '#009688',
              'polarization_drift': '#607D8B',
              'optical_jamming': '#F44336', 'intercept_resend': '#E91E63',
              'yield_manipulation': '#FF5722', 'detector_control': '#FF9800'}
bars = class_counts.sort_index()
ax.barh(range(len(bars)), bars.values,
        color=[colors_map.get(s, '#999') for s in bars.index])
ax.set_yticks(range(len(bars)))
ax.set_yticklabels(bars.index)
ax.set_xlabel('Number of windows')
ax.set_title('Telemetry Dataset: Class Distribution (30 passes)')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()
plt.close()

---
## 8. Physics-Informed Feature Extraction

The `features` module extracts **24 physics-informed features** from the raw telemetry, organised in four source groups:
- **Quantum statistics (q)**: loss/QBER/vacuum residuals, decoy and basis deviations, rates of change, rolling statistics, SNR
- **Terminal telemetry (t)**: beacon power ratios, pointing residual, scintillation
- **Receiver monitors (r)**: out-of-gate, double-click rates, polarisation reference
- **Derived**: elevation, atmospheric transmittance deviation, anomaly distance (fixed-scale RMS of physical residuals; label-free)

In [ ]:
# Feature dictionary
fd = feature_dictionary()
print(f"=== Feature Registry ({len(fd)} features) ===")
fd[['name', 'group', 'unit', 'quantum_only']]

In [ ]:
# Extract features and show per-class statistics for top features
X = extract_features(dataset, link)
y = dataset['label'].values
print(f"Feature matrix shape: {X.shape}")

top_feats = ['loss_residual', 'qber_residual', 'vacuum_rate_residual',
             'decoy_ratio_deviation', 'beacon_power_ratio', 'anomaly_distance']
top_idx = [FEATURE_NAMES.index(f) for f in top_feats]

rows = []
for label in sorted(set(y)):
    mask = y == label
    for fi, fname in zip(top_idx, top_feats):
        rows.append(dict(label=label, feature=fname,
                         mean=X[mask, fi].mean(), std=X[mask, fi].std()))

df_stats = pd.DataFrame(rows).pivot_table(index='label', columns='feature',
                                           values='mean').round(3)
print("\n=== Mean Feature Value per Class (selected features) ===")
df_stats[top_feats]

---
## 9. ML Classification Pipeline

The `models` module provides three multi-class classifiers and two novelty detectors for telemetry-based anomaly detection:

- **Classifiers**: Gradient Boosting, Random Forest, Logistic Regression — all with StandardScaler preprocessing
- **Novelty detectors**: Isolation Forest, Mahalanobis distance — trained on nominal data only

The pipeline: raw telemetry → 24 features → scaler → classifier → per-window label → pass-level alarm.

In [ ]:
# Independent training, validation and test passes (full classifier comparison).
ev = full_evaluation(link, n_train=30, n_test=20, seed=42)
train_df, test_df = ev['train_df'], ev['test_df']
X_train, X_test = ev['X_train'], ev['X_test']
y_train, y_test = ev['y_train'], ev['y_test']
results = ev['classifiers']
for method, data in results.items():
    metrics = data['metrics']
    print(f"{method:25s} macro_F1 = {metrics['macro_f1']:.3f} accuracy = {metrics['accuracy']:.3f}")
best_method = ev['best_method']
print(f"Selected on independent validation passes: {best_method}")

In [ ]:
# Confusion matrix for the best classifier
best_metrics = results[best_method]['metrics']
cm = best_metrics['confusion_matrix']
labels = best_metrics['labels']

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(labels, fontsize=8)

for i in range(len(labels)):
    for j in range(len(labels)):
        val = cm[i, j]
        color = 'white' if val > cm.max() * 0.5 else 'black'
        ax.text(j, i, str(val), ha='center', va='center', color=color, fontsize=7)

ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — {best_method} (macro F1 = {best_metrics["macro_f1"]:.3f})')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# Feature importance (permutation-based, top 10)
best_model = results[best_method]['model']
fi = feature_importance(best_model, X_test, y_test, n_repeats=5, seed=42)

fi_df = pd.DataFrame(dict(feature=FEATURE_NAMES,
                           importance=fi['importances_mean'],
                           std=fi['importances_std']))
fi_df = fi_df.sort_values('importance', ascending=True).tail(10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(fi_df['feature'], fi_df['importance'], xerr=fi_df['std'],
        color='#2196F3', alpha=0.8)
ax.set_xlabel('Permutation Importance (macro F1 decrease)')
ax.set_title(f'Top 10 Features — {best_method}')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# Novelty detectors: train on nominal data only
nom_mask = y_train == 'nominal'
print(f"Nominal training samples: {nom_mask.sum()}")

for nd_method in ['isolation_forest', 'mahalanobis']:
    det = train_novelty_detector(X_train[nom_mask], method=nd_method, seed=42)
    is_anom = novelty_predict(det, X_test)
    tp = ((y_test != 'nominal') & is_anom).sum()
    fn = ((y_test != 'nominal') & ~is_anom).sum()
    fp = ((y_test == 'nominal') & is_anom).sum()
    tn = ((y_test == 'nominal') & ~is_anom).sum()
    det_rate = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    print(f"\n{nd_method}:")
    print(f"  Detection rate: {det_rate:.3f}  |  False positive rate: {fpr:.3f}")
    print(f"  TP={tp}, FN={fn}, FP={fp}, TN={tn}")

---
## 10. Evaluation Framework & Conclusions

The `evaluation` module orchestrates the complete experimental pipeline: dataset generation, feature extraction, classifier training, novelty detection, alarm logic, and block-policy analysis. It also includes ablations: static-threshold baselines, leave-one-attack-out robustness, and shifted-hardware generalisation tests.

In [ ]:
# Reuse the completed full evaluation; no duplicate model training.
st = summary_table(ev)
print('Classifier summary on independent test data')
st.round(3)

In [ ]:
# Block policies: key yield under different exclusion strategies
y_pred = predict(results[best_method]['model'], X_test)
key_w = test_df['key_bits'].values

rows = []
for pol in ['exclude_jamming', 'exclude_all_attacks', 'exclude_all_anomalies']:
    bp = block_policy(y_pred, key_w, policy=pol)
    rows.append(dict(policy=pol, total_key_Mbit=bp['total_key']/1e6,
                     retained_key_Mbit=bp['retained_key']/1e6,
                     retained_pct=bp['retained_fraction']*100))
df_bp = pd.DataFrame(rows)
print("=== Block Policy Analysis ===")
df_bp.round(2)

---
## Conclusions

1. **Link model:** the baseline gives about 34.8 dB loss and 23.8 kbit/s at 90°, versus 43.2 dB and 2.8 kbit/s at 20°, for the stated 500 km link. These regression values are model calculations.
2. **Finite-size model:** the table above shows the actual estimates after including signal errors and intensity allocation. No operational security guarantee follows from these values.
3. **OPA:** the example illustrates steering, first-null width and phase loss. The default array admits grating lobes; the principal angular interval is not a guarantee of their absence.
4. **GEO:** a four-node ring covers approximately 84.6% of Earth's surface at least once and 21.1% twice above 20°. Adjacent ring nodes are approximately 59,619 km apart. Geometry alone does not establish feasibility of a quantum link.
5. **Key budget:** generation, consumption, capacity and underflow determine availability. Inspect the simulated deficit rather than assuming continuous stable operation.
6. **Telemetry and ML:** training, selection and testing use separate passes. Read the measured per-class scores and confusion matrix; accuracy on synthetic signatures does not establish real-system performance. Anomaly-distance features use no ground-truth test labels.

The outputs are a reproducible research demonstration. Hardware validation, complete protocol security proofs and external experimental data are outside this project's scope. See `README_UK.md` and `VERIFICATION.md`.

In [ ]:
# Package info
import importlib, pkgutil
print(f"satqkd_sentinel v{satqkd_sentinel.__version__}\n")
print("=== Submodules ===")
for importer, name, ispkg in pkgutil.walk_packages(
        satqkd_sentinel.__path__, satqkd_sentinel.__name__ + '.'):
    mod = importlib.import_module(name)
    doc = (mod.__doc__ or '').strip().split('\n')[0]
    print(f"  {name.split('.')[-1]:15s}  {doc}")

## Перевірка всіх модулів, експорт і завантаження


In [ ]:
# Автоматична перевірка всього проєкту в поточному середовищі.
import json, time
_test_env = dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD="1", PYTHONPATH=str(PROJECT_ROOT), MPLBACKEND="Agg")
_test_start = time.perf_counter()
_test_run = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"],
                           cwd=PROJECT_ROOT, env=_test_env, capture_output=True, text=True)
_test_log = _test_run.stdout + _test_run.stderr
(OUTPUT_DIR / "tests.txt").write_text(_test_log, encoding="utf-8")
print(_test_log)
assert _test_run.returncode == 0, "Є невдалі тести — див. tests.txt; експорт зупинено."
assert np.isfinite(X_train).all() and np.isfinite(X_test).all()
assert len(list(OUTPUT_DIR.glob("figure_*.png"))) == 13
TEST_STATUS = {"returncode": _test_run.returncode, "seconds": round(time.perf_counter()-_test_start,2)}
print("Перевірки пройдено; 13 графіків збережено.")


In [ ]:
# Експорт уже обчислених результатів, без повторного навчання.
import html, joblib, shutil
from dataclasses import asdict
from satqkd_sentinel.cli import versions
summary_table(ev).to_csv(OUTPUT_DIR / "metrics.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "telemetry.csv", index=False)
_predictions = test_df[["pass_id", "t_s", "label", "key_bits"]].copy()
_predictions["prediction"] = predict(results[best_method]["model"], X_test)
_predictions.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
for name, frame in [("finite_key",df_fk),("geo_coverage",df_cov),("isl",df_isl),
                    ("daily_budget",df_budget),("block_policy",df_bp)]:
    frame.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
joblib.dump({"version":satqkd_sentinel.__version__,"sklearn_version":importlib.metadata.version("scikit-learn"),
             "feature_names":FEATURE_NAMES,"model":results[best_method]["model"]},OUTPUT_DIR / "model.joblib")
_run_info = {"version":satqkd_sentinel.__version__,"seed":42,"train_passes":30,"test_passes":20,
             "selected_method":best_method,"selection_basis":ev["selection_basis"],"tests":TEST_STATUS,
             "environment":versions(),"link_parameters":asdict(link),
             "classifiers":{name:{"macro_f1":float(data["metrics"]["macro_f1"]),
                                  "accuracy":float(data["metrics"]["accuracy"])} for name,data in results.items()}}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(_run_info,ensure_ascii=False,indent=2,allow_nan=False),encoding="utf-8")
_images = "".join('<img style="width:100%;height:auto;margin:12px 0" src="data:image/png;base64,' +
                  base64.b64encode(p.read_bytes()).decode() + '" alt="'+p.stem+'">'
                  for p in sorted(OUTPUT_DIR.glob("figure_*.png")))
_report = ('<!doctype html><html lang="uk"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">'
           '<title>SatQKD — Colab results</title><body style="max-width:1100px;margin:24px auto;padding:16px;font:16px system-ui">'
           '<h1>SatQKD-Sentinel 2.0.2 — результати</h1><p>Синтетичні дані. Модель обрана на окремій валідаційній вибірці.</p>'
           + summary_table(ev)[["method","macro_f1","accuracy"]].round(4).to_html(index=False)
           + '<h2>Автоматичні перевірки</h2><pre style="white-space:pre-wrap">'+html.escape(_test_log)+'</pre>'
           + _images + '</body></html>')
(OUTPUT_DIR / "report.html").write_text(_report,encoding="utf-8")
RESULTS_ZIP = Path(shutil.make_archive(str(PROJECT_ROOT / "satqkd_colab_results"), "zip", OUTPUT_DIR))
print("Результати:", RESULTS_ZIP)
print("Розмір ZIP:", round(RESULTS_ZIP.stat().st_size/1024/1024,2), "MiB")


In [ ]:
# Завантаження запускається лише натисканням кнопки, не блокує Run all.
from IPython.display import display, FileLink
try:
    from google.colab import files as _colab_files
except ImportError:
    display(FileLink(str(RESULTS_ZIP)))
else:
    import ipywidgets as widgets
    _download_button = widgets.Button(description="Завантажити результати", button_style="success", layout=widgets.Layout(width="260px"))
    _download_output = widgets.Output()
    def _download_results(_):
        with _download_output:
            _colab_files.download(str(RESULTS_ZIP))
    _download_button.on_click(_download_results)
    display(_download_button, _download_output)
